# Backpropagation by Hand

This notebook computes the same gradients in two ways: PyTorch's automatic differentiation and an explicit application of the chain rule. The network has two input features, one hidden unit, and one scalar output.

Run the cells in order with PyTorch installed. One final gradient-descent update shows how gradients change predictions; this is one optimization step, not a full training run.


## Inputs, Parameters, and Learning Rate

The input has shape <code>(1, 2)</code> and the target has shape <code>(1, 1)</code>: one example with two features and one response. <code>requires_grad=True</code> tells PyTorch to track operations on the learnable weights and biases.

The hidden weight matrix has shape <code>(2, 1)</code>; the output weight has shape <code>(1, 1)</code>. Biases are added by broadcasting. The seed reproduces initialization, and the learning rate sets the size of the eventual update.


In [ ]:
import torch

# Set random seed for reproducibility
torch.manual_seed(42)

# Define input (2 features) and target output (scalar)
X = torch.tensor([[0.5, 1.0]], dtype=torch.float32)  # Input shape: (1,2)
y = torch.tensor([[1.0]], dtype=torch.float32)       # Target shape: (1,1)

# Initialize weights and bias
w1 = torch.randn((2, 1), requires_grad=True)  # Weights for input layer (2 → 1 hidden unit)
b1 = torch.zeros(1, requires_grad=True)       # Bias for hidden layer

w2 = torch.randn((1, 1), requires_grad=True)  # Weights for hidden layer → output
b2 = torch.zeros(1, requires_grad=True)       # Bias for output layer

# Learning rate
lr = 0.1



## Hidden Layer

First compute $z_1=Xw_1+b_1$. The **rectified linear unit (ReLU)** activation keeps positive values and replaces negative ones with zero: $a_1=\max(0,z_1)$.

For this one-unit network, both intermediate tensors have shape <code>(1, 1)</code>.


In [ ]:
# Step 1: Forward Pass (Layer 1 - Hidden Layer)
z1 = X @ w1 + b1  # Linear transformation
a1 = torch.relu(z1)  # Activation function (ReLU)



## Output and Loss

The output layer is linear: $O=a_1w_2+b_2$. The code uses **mean squared error (MSE)** with no factor of one half. There is one scalar prediction, so $L=(O-y)^2$ and $\partial L/\partial O=2(O-y)$.

The printed prediction and loss are measured before any parameter update.


In [ ]:
# Step 2: Forward Pass (Layer 2 - Output Layer)
z2 = a1 @ w2 + b2  # Linear transformation
output = z2  # Final output

# Step 3: Compute Loss (Mean Squared Error)
loss = torch.mean((output - y) ** 2)

# Print forward pass results
print(f"Output before backpropagation: {output.item():.4f}")
print(f"Loss before backpropagation: {loss.item():.4f}")



Output before backpropagation: 0.0697
Loss before backpropagation: 0.8655


## Ask Autograd for Gradients

<code>loss.backward()</code> traverses the computation graph and accumulates derivatives in each leaf parameter's <code>.grad</code> field. It does not change parameter values. We retain these gradients to compare against the manual calculation.


In [ ]:
# Step 4: Backward Pass (Compute Gradients)
loss.backward()  # Automatically computes gradients using backpropagation


## Compute the Chain Rule Manually

Let $\delta_2=2(O-y)$. The output is linear, so

$$\frac{\partial L}{\partial w_2}=a_1\delta_2, \qquad \frac{\partial L}{\partial b_2}=\delta_2.$$

The hidden error is $\delta_1=\delta_2 w_2\,\mathbb{1}[z_1>0]$, giving

$$\frac{\partial L}{\partial w_1}=X^\top\delta_1, \qquad \frac{\partial L}{\partial b_1}=\delta_1.$$

PyTorch uses derivative zero for ReLU at zero. The code's Boolean mask follows that convention. <code>torch.no_grad()</code> prevents these manual calculations from creating another autograd graph.

The elementwise products below work for this single-example, single-hidden-unit network. Larger batches and layers require appropriate matrix products and sums over examples.


In [ ]:
with torch.no_grad():
    # Step 5: Compute Manual Gradients
    dL_dO = 2 * (output - y) / 1  # Derivative of MSE loss w.r.t. output
    dO_dz2 = 1  # Since output = z2, its derivative is 1
    dL_dz2 = dL_dO * dO_dz2  # Chain rule

    # Gradients for w2 and b2 (Output layer)
    dL_dw2 = dL_dz2 * a1  # dL/dw2 = dL/dz2 * da1/dw2
    dL_db2 = dL_dz2 * 1    # dL/db2 = dL/dz2 * db2/db2

    # Gradients for w1 and b1 (Hidden layer)
    dz2_da1 = w2  # Derivative of z2 w.r.t. a1 (weights from hidden to output layer)
    da1_dz1 = (z1 > 0).float()  # ReLU derivative (1 if z1 > 0, else 0)

    dL_dz1 = dL_dz2 * dz2_da1 * da1_dz1  # Chain rule: dL/dz1 = dL/dz2 * dz2/da1 * da1/dz1
    dL_dw1 = dL_dz1 * X.T  # dL/dw1 = dL/dz1 * X
    dL_db1 = dL_dz1 * 1  # dL/db1 = dL/dz1 * db1/db1



## Compare the Four Parameter Gradients

Compare the automatic and manual entries for both weights and both biases. The first-layer weight gradient has two entries because the hidden unit receives two features. The saved combined log is displayed after the final forward pass below.


In [ ]:
# Print Gradients
print("\nAutomatic Gradients (PyTorch):")
print(f"dL/dw1: {w1.grad}")
print(f"dL/db1: {b1.grad}")
print(f"dL/dw2: {w2.grad}")
print(f"dL/db2: {b2.grad}")

print("\nManual Gradients (Calculated using Chain Rule):")
print(f"dL/dw1: {dL_dw1}")
print(f"dL/db1: {dL_db1}")
print(f"dL/dw2: {dL_dw2}")
print(f"dL/db2: {dL_db2}")



## Check Numerical Agreement

<code>torch.allclose</code> checks agreement within floating-point tolerances. All four checks should be <code>True</code>. A missing factor of two in the MSE derivative would fail this comparison.

Some manual bias gradients retain a <code>(1, 1)</code> shape while the parameter gradients have shape <code>(1,)</code>; this comparison allows broadcasting. Values agree here, but shape checks also matter when extending the network.


In [ ]:
# Step 6: Verify Matching
print("\nChecking if manual and automatic gradients match:")
print(torch.allclose(w1.grad, dL_dw1), "for dL/dw1")
print(torch.allclose(b1.grad, dL_db1), "for dL/db1")
print(torch.allclose(w2.grad, dL_dw2), "for dL/dw2")
print(torch.allclose(b2.grad, dL_db2), "for dL/db2")



## Take a Gradient-Descent Step

Each parameter follows $\theta\leftarrow\theta-\eta\nabla_\theta L$. The update occurs inside <code>no_grad()</code>, since changing the parameters is not part of the next differentiation graph.

The gradients are then zeroed because PyTorch accumulates them by default. Rerunning the comparison after this cell would compare against zeroed gradients; restart from initialization for a fresh full comparison.


In [ ]:
# Step 7: Update Weights (Gradient Descent)
with torch.no_grad():
    w1 -= lr * w1.grad
    b1 -= lr * b1.grad
    w2 -= lr * w2.grad
    b2 -= lr * b2.grad

    # Zero out gradients after update
    w1.grad.zero_()
    b1.grad.zero_()
    w2.grad.zero_()
    b2.grad.zero_()



## Recompute the Prediction

Run the forward equations again with the updated parameters and measure a new loss. The saved log contains the earlier gradient values, the four agreement checks, and the prediction and loss after the update.

A smaller loss in this example confirms that the chosen update helped on this example. An excessively large learning rate can increase loss even when the gradient is correct.


In [ ]:
# Step 8: Forward Pass Again (After Weight Update)
z1 = X @ w1 + b1
a1 = torch.relu(z1)
z2 = a1 @ w2 + b2
new_output = z2

# Compute new loss
new_loss = torch.mean((new_output - y) ** 2)

# Print updated results
print(f"\nOutput after weight update: {new_output.item():.4f}")
print(f"Loss after weight update: {new_loss.item():.4f}")


Automatic Gradients (PyTorch):
dL/dw1: tensor([[-0.2181],
        [-0.4363]])
dL/db1: tensor([-0.4363])
dL/dw2: tensor([[-0.5529]])
dL/db2: tensor([-1.8607])

Manual Gradients (Calculated using Chain Rule):
dL/dw1: tensor([[-0.2181],
        [-0.4363]])
dL/db1: tensor([[-0.4363]])
dL/dw2: tensor([[-0.5529]])
dL/db2: tensor([[-1.8607]])

Checking if manual and automatic gradients match:
True for dL/dw1
True for dL/db1
True for dL/dw2
True for dL/db2

Output after weight update: 0.3006
Loss after weight update: 0.4891


## Check Your Understanding

- Which operations compute gradients, which modify weights, and which clear accumulated gradients?
- If the hidden preactivation were negative, which weight gradients would become zero? Could the output bias still learn?
- Why does MSE averaged over several outputs require dividing its derivative by the number of averaged elements?

<details class="notebook-answers">
<summary>Answers and discussion</summary>

1. **Gradient calculation, parameter update, and reset are separate.** `loss.backward()` accumulates automatic gradients in `.grad`; the explicit chain-rule expressions compute the manual versions. The statements such as `w1 -= lr * w1.grad` change parameters. Calls such as `w1.grad.zero_()` clear the stored gradients. `torch.no_grad()` only disables graph tracking for operations inside its block; it neither clears gradients nor updates parameters by itself.
2. **A negative hidden preactivation blocks this unit's gradient.** ReLU gives `a1 = 0` and derivative zero. Both entries of the `w1` gradient and the `b1` gradient become zero. The `w2` gradient is also zero because its input `a1` is zero. The output bias still has gradient $2(O-y)$, so it can learn unless the prediction already matches the target.
3. **Differentiation preserves the averaging factor.** For m scalar output elements, MSE is the sum of their squared errors divided by m. The derivative for element i is therefore $2(O_i-y_i)/m$. Omitting the division differentiates the sum of squared errors instead, scaling the gradient by m and making the update magnitude depend on how many elements were averaged. Here m is one.

</details>

**Takeaway:** backpropagation computes derivatives; gradient descent uses them. Keeping those two steps separate makes training code easier to debug.
